# Group A ASR Full Run
This notebook runs the complete HW14 Group A ASR workflow in Colab. It is intended for H100 or A100 execution after the dry run succeeds.

What this notebook does:
- Mounts Google Drive and resolves the project root
- Installs required packages
- Executes the full Group A workflow with checkpointing
- Runs the Step 3 verification script
- Packages the Group A outputs, logs, and checkpoint snapshot into a ZIP bundle

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%pip install -q transformers datasets accelerate genaibook jiwer scipy soundfile numpy pandas matplotlib sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 26.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 153.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 82.0 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import os

import sys



DRIVE_ROOT = Path('/content/drive/MyDrive')

CANDIDATE_ROOTS = [

    DRIVE_ROOT / 'kamp_hw14',

    DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw14',

    DRIVE_ROOT / 'GEN_AI' / 'HW1' / 'kamp_hw14',

]



def find_project_root():

    checked_paths = []

    for candidate in CANDIDATE_ROOTS:

        checked_paths.append(candidate)

        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():

            return candidate



    discovered_roots = []

    for match in DRIVE_ROOT.rglob('kamp_hw14.py'):

        if match.parent.name != 'hw14_scripts':

            continue

        candidate = match.parent.parent

        discovered_roots.append(candidate)

        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():

            return candidate



    checked_display = '\n'.join(f' - {path}' for path in checked_paths)

    discovered_display = '\n'.join(f' - {path}' for path in discovered_roots[:10]) or ' - none found'

    raise FileNotFoundError(

        'Could not locate kamp_hw14 on Google Drive.\n'

        'Upload the project to /content/drive/MyDrive/kamp_hw14\n'

        'or keep any Drive copy that contains hw14_scripts/kamp_hw14.py.\n'

        f'Checked:\n{checked_display}\n'

        f'Discovered matches:\n{discovered_display}'

    )



PROJECT_ROOT = find_project_root()

os.chdir(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT / 'hw14_scripts'))



print(f'PROJECT_ROOT = {PROJECT_ROOT}')

print(f'Module exists: {(PROJECT_ROOT / "hw14_scripts" / "kamp_hw14.py").exists()}')

PROJECT_ROOT = /content/drive/MyDrive/kamp_hw14
Module exists: True


In [4]:
from contextlib import redirect_stderr, redirect_stdout

from datetime import datetime

import importlib

import shutil

import subprocess



import kamp_hw14

kamp_hw14 = importlib.reload(kamp_hw14)



from kamp_hw14 import run_group_a_asr

from hw14_data_utils import CHECKPOINT_PATH, TeeLogger, load_checkpoint, save_checkpoint, save_text_artifact



GROUP_A_ROOT = PROJECT_ROOT / 'hw14_experiments' / 'group_a_asr'

FULL_LOG_PATH = PROJECT_ROOT / 'hw14_printouts' / 'group_a_asr_full_log.txt'

FULL_SUMMARY_PATH = GROUP_A_ROOT / 'full' / 'group_a_full_summary.json'

VERIFY_SCRIPT = PROJECT_ROOT / 'hw14_scripts' / 'verification_scripts' / 'verify_group_a.py'



GROUP_A_ROOT.mkdir(parents=True, exist_ok=True)

FULL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)



print(f'kamp_hw14 imported from: {kamp_hw14.__file__}')

print(f'GROUP_A_ROOT = {GROUP_A_ROOT}')

print(f'FULL_LOG_PATH = {FULL_LOG_PATH}')

kamp_hw14 imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/kamp_hw14.py
GROUP_A_ROOT = /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_a_asr
FULL_LOG_PATH = /content/drive/MyDrive/kamp_hw14/hw14_printouts/group_a_asr_full_log.txt


In [7]:
summary = None

with TeeLogger(FULL_LOG_PATH) as tee, redirect_stdout(tee), redirect_stderr(tee):

    print('[NOTEBOOK] Starting Group A full run')

    summary = run_group_a_asr(mode='full')

    print(summary)



save_text_artifact(FULL_SUMMARY_PATH, summary, as_json=True)

print(summary)

[NOTEBOOK] Starting Group A full run
Device set to use cuda:0
[timer] ga20a_baseline: 0.495s
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[timer] ga20b_baseline: 0.662s
[timer] ga20c_baseline: 2.258s
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[timer] ga20d_corrected_baseline: 13.413s
[timer] ga

In [8]:
subprocess.run([sys.executable, str(VERIFY_SCRIPT)], check=True)

CompletedProcess(args=['/usr/bin/python3', '/content/drive/MyDrive/kamp_hw14/hw14_scripts/verification_scripts/verify_group_a.py'], returncode=0)

In [9]:
bundle_root = PROJECT_ROOT / 'hw14_experiments' / '_group_a_bundle'
if bundle_root.exists():
    shutil.rmtree(bundle_root)

logs_dir = bundle_root / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)
group_bundle_root = bundle_root / 'group_a_asr'
shutil.copytree(GROUP_A_ROOT, group_bundle_root, dirs_exist_ok=True)

checkpoint_snapshot = load_checkpoint(CHECKPOINT_PATH)
(group_bundle_root / 'full').mkdir(parents=True, exist_ok=True)
save_checkpoint(checkpoint_snapshot, group_bundle_root / 'full' / 'checkpoint_snapshot.json')

for extra_path in [FULL_LOG_PATH, PROJECT_ROOT / 'hw14_printouts' / 'kamp_hw14_execution_log.txt']:
    if extra_path.exists():
        shutil.copy2(extra_path, logs_dir / extra_path.name)

if FULL_SUMMARY_PATH.exists():
    shutil.copy2(FULL_SUMMARY_PATH, group_bundle_root / 'full' / FULL_SUMMARY_PATH.name)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_base = PROJECT_ROOT / 'hw14_experiments' / f'group_a_results_{timestamp}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=bundle_root)
print(f'Created ZIP: {archive_path}')

Created ZIP: /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_a_results_20260404_004815.zip


In [10]:
try:
    from google.colab import files
    files.download(archive_path)
except Exception as error:
    print(f'Download step skipped: {error}')
    print(f'Manually download: {archive_path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>